## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from imblearn.combine import SMOTETomek


import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [2]:
dagshub.init(
    repo_owner="Yashwanth-R19",
    repo_name="Boston-Housing-MLFlow",
    mlflow=True
)

Accessing as Yashwanth-R19

Repository Boston-Housing-MLFlow doesn't exist, creating it under current user.

Initialized MLflow to track repo "Yashwanth-R19/Boston-Housing-MLFlow"

Repository Yashwanth-R19/Boston-Housing-MLFlow initialized!

In [3]:
mlflow.set_experiment(
    "Boston Housing Regression PBLM 1"
)

2026/08/07 10:22:23 INFO mlflow.tracking.fluent: Experiment with name 'Boston Housing Regression PBLM 1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/07cc764e2fd44da8824642dd8d295f4f', creation_time=1786078345334, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786078345334, lifecycle_stage='active', name='Boston Housing Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

## Data Loading and Processing

In [4]:
import pandas as pd

df = pd.read_csv("BostonHousing.csv")

X = df.drop("medv", axis=1)
y = df["medv"]

print(X.shape)
print(y.shape)

(506, 13)
(506,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [6]:
X_train_res = X_train.copy()
y_train_res = y_train.copy()

print(X_train_res.shape)
print(y_train_res.shape)

(354, 13)
(354,)


## Build Models

In [7]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest Regressor",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost Regressor",
        XGBRegressor(
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [8]:
print(df.isnull().sum())

crim       0
zn         0
indus      0
chas       0
nox        0
rm         0
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64


In [9]:
reports = []
trained_models = []

for model_name, model, X_tr, y_tr in models:
    model.fit(X_tr, y_tr)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    report = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

Linear Regression
MAE  : 3.1627
MSE  : 21.5174
RMSE : 4.6387
R2   : 0.7112
Random Forest Regressor
MAE  : 2.2757
MSE  : 11.1563
RMSE : 3.3401
R2   : 0.8503
XGBoost Regressor
MAE  : 2.0600
MSE  : 9.2880
RMSE : 3.0476
R2   : 0.8754


## Log All Experiments to DagsHub

In [10]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Log model name
        mlflow.log_param(
            "Model",
            model_name
        )

        # Log hyperparameters
        mlflow.log_params(
            model.get_params()
        )

        # Log regression metrics
        mlflow.log_metric(
            "MAE",
            report["MAE"]
        )

        mlflow.log_metric(
            "MSE",
            report["MSE"]
        )

        mlflow.log_metric(
            "RMSE",
            report["RMSE"]
        )

        mlflow.log_metric(
            "R2",
            report["R2"]
        )

        # Log model
        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged!")

2026/08/07 10:22:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0/runs/e1bf08c1b9db48bdbee445eddde64276
🧪 View experiment at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0


2026/08/07 10:23:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0/runs/8fa6eb64bd6a4c64a75e02de3f8a63ed
🧪 View experiment at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0


2026/08/07 10:24:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Regressor at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0/runs/c05d59f46dd24e80b858aeb635cf4b6f
🧪 View experiment at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0
Experiments logged!


## Best Model and Reg to DH

In [11]:
best_index = np.argmax(
    [
        r["R2"]
        for r in reports
    ]
)

best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print("Best Model:", best_model_name)
print("R2 Score:", best_report["R2"])

Best Model: XGBoost Regressor
R2 Score: 0.8753514389204832


In [12]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    else:

        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    run_id = run.info.run_id

print(run_id)

2026/08/07 10:24:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Boston_Housing_Best_Model'.
2026/08/07 10:25:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Housing_Best_Model, version 1
Created version '1' of model 'Boston_Housing_Best_Model'.


🏃 View run Champion_XGBoost Regressor at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0/runs/db21e3bf3b87405190e15e96b543bb45
🧪 View experiment at: https://dagshub.com/Yashwanth-R19/Boston-Housing-MLFlow.mlflow/#/experiments/0
db21e3bf3b87405190e15e96b543bb45
